## Functions : 
- A function is a reusable block of code that does one specific job. You define it once, call it anywhere.
- Example : 
def greet() -> str : # Function defination
    return("Hello from your first function!")

print(greet())   # Function calling 

Four parts to every function:
1. 'def' keyword — tells Python you are defining a function
2. Function name - Like 'greet' above
3. () — where inputs go (empty means no inputs needed)
4.  -> : What the function returns

Another example : 
    def call_llm(prompt: str, **kwargs) -> dict:  # 'prompt: str' means prompt should be a string type

## Part 1 : Args & Kwargs
Types : 
1. Regular Arguments (args)
    - Basic inputs we pass into a function.
    - Python matches arguments by position — first value goes to first argument, second to second.

2. Default Arguments
    - Default args are used in Function DEFINATION. 
    - Give arguments a default value — makes them optional when calling
    - Rule — default arguments always go after non-default ones

3. Keyword Arguments (Kwargs)
   - Keyword args are used when CALLING the function. They are a different calling style.
   - In AI engineering, keyword arguments are used constantly when calling LLM APIs

4. *args — Variable Positional Arguments
    - When you don't know how many arguments will be passed — use *args. 
    - It collects all extra positional arguments into a tuple.
    - The * is the important part — args is just the conventional name. You could write *scores and it works the same.

5. **kwargs — Variable Keyword Arguments
    - When you don't know what named arguments will be passed — use **kwargs. 
    - It collects all extra keyword arguments into a dict.
    - Again ** is the important part — kwargs is just the conventional name.    

(See example code below)

[Question] In kwargs can the function defination and function call have different names ?
[Response] 
1. With regular named arguments — names must MATCH
def evaluate(score, threshold):
    < some code >

evaluate(score=0.9, threshold=0.8)    # ✅ names match
evaluate(points=0.9, threshold=0.8)   # ❌ TypeError — "points" not recognized

2. With **kwargs — there are NO fixed names in definition
def build_config(**kwargs):  # no fixed names here — accepts ANYTHING

build_config(model="gpt-4", temperature=0.7)
build_config(prompt="hello", max_tokens=512)
build_config(anything="you want", foo="bar")

## Part 2 : 
1. Basic Return
    - 'return' sends a value back to whoever called the function.
    -  Without 'return' — function returns 'None'.

2. Returning Multiple Values
    - Python lets you return multiple values — wrapped automatically as a tuple.

3. Returning a Dict — Most Common in AI Work
    - When returning many values, a dict is cleaner than a long tuple 

4. Early Return — Exit Function Early
    - 'return' can be used to exit a function early when a condition is met.
    - This pattern — checking for bad input at the top and returning early — is called a guard clause. 
    - We will use it constantly in eval pipelines.    

## Part 3 : try/except
- LLM APIs fail. JSON is malformed. Files don't exist. Network times out. Without error handling your entire eval pipeline crashes — losing all previous results.
- 'try/except' lets us handle errors gracefully instead of crashing.

try:
    risky_code()
except SpecificError:
    handle_it()
except SpecificError as e:
    print(e)
except Exception as e:
    print(f"unexpected: {e}")
else:
    print("success")
finally:
    print("always runs")

[Question] What does 'else' and 'finally' do ? 
[Response] 
    1. 'else' runs only when NO exception occurred. It runs only when 'try' succeeded. Keeps success logic separate from risky code.
    2. 'finally' runs regardless of outcome. It has Cleanup code that must happen like closing files, closing database connections, logging etc.

 Real AI Eval Pattern — Full Error Handling : See code below

In [ ]:
# Functions : Example Code for the Notes above.

## Part 1 : Args & Kwargs
# 1. Regular Arguments (args)
   
def evaluate(score, threshold):      # score and threshold are arguments
    return score >= threshold

evaluate(0.9, 0.8)    # score=0.9, threshold=0.8

# 2. Default Arguments
def evaluate(score, threshold=0.8):   # threshold has a default. Default arguments always go after non-default ones
    return score >= threshold

evaluate(0.9)          # uses default threshold=0.8 → True
evaluate(0.9, 0.5)     # overrides default → True
evaluate(0.4, 0.8)     # overrides default → False

# 3. Keyword Arguments (Kwargs)
def evaluate(score, threshold=0.8, model="unknown"):
    return score >= threshold

# Positional — must be in order
evaluate(0.9, 0.8, "gpt-4")

# Keyword — any order, self documenting
evaluate(score=0.9, model="gpt-4", threshold=0.8)

# 4. *args — Variable Positional Arguments
def average_scores(*args): # args is just a conventional name. You could write *scores or anything else
    print(args)           # tuple of everything passed in
    return sum(args) / len(args)

average_scores(0.9, 0.4, 0.85)         # args = (0.9, 0.4, 0.85)
average_scores(0.9, 0.4, 0.85, 0.6)    # args = (0.9, 0.4, 0.85, 0.6)

# 5. **kwargs — Variable Keyword Arguments
def build_config(**kwargs):
    print(kwargs)    # dict of everything passed in
    return kwargs

build_config(model="gpt-4", temperature=0.7, max_tokens=512)
# kwargs = {"model": "gpt-4", "temperature": 0.7, "max_tokens": 512}

# 5.0 Combining Everything - Regular args + *args + Keyword args with defaults + **kwargs
def run_eval(model, *case_ids, threshold=0.8, **config):
    print(f"Model: {model}")
    print(f"Cases: {case_ids}")
    print(f"Threshold: {threshold}")
    print(f"Config: {config}")

run_eval(
    "gpt-4",
    "case_1", "case_2", "case_3",
    threshold=0.9,
    temperature=0.7,
    max_tokens=512
)
# Model: gpt-4
# Cases: ('case_1', 'case_2', 'case_3')
# Threshold: 0.9
# Config: {'temperature': 0.7, 'max_tokens': 512}

# Note : 
# -  In kwargs with regular named arguments — function defination and function call names must MATCH
# - In kwargs with **kwargs — there are NO fixed names in definition

## Real AI Eval Pattern with kwargs
def call_llm(prompt: str, **kwargs) -> dict:
    config = {
        "model":       kwargs.get("model", "gpt-4"),
        "temperature": kwargs.get("temperature", 0.7),
        "max_tokens":  kwargs.get("max_tokens", 512),
    }
    print(f"Calling {config['model']} with prompt: {prompt}")
    return config

call_llm("What is 2+2?", model="claude-3", temperature=0.2)

# -------------------------------------------------------------- #
## Part 2 : Return 

# 1. Basic Return
def evaluate(score: float) -> bool:
    return score >= 0.8

result = evaluate(0.9)
print(result)    # True

# 2. Returning Multiple Values
def evaluate(response: dict) -> tuple:
    score  = response.get("score", 0)
    passed = score >= 0.8
    reason = response.get("reason", "none")
    return score, passed, reason       # returns tuple automatically

score, passed, reason = evaluate(response)   # unpack immediately

# 3. Returning a Dict — Most Common in AI Work
def evaluate(response: dict) -> dict:
    score = response.get("score", 0)
    return {
        "score":   score,
        "passed":  score >= 0.8,
        "model":   response.get("model", "unknown"),
        "reason":  response.get("reason", "none")
    }

result = evaluate({"model": "gpt-4", "score": 0.91})
print(result["passed"])    # True
print(result["model"])     # "gpt-4"

# 4. Early Return — Exit Function Early
def evaluate(response: dict) -> dict:

    # guard clause — exit early if response is empty
    if not response:
        return {"error": "empty response", "passed": False}

    score = response.get("score", 0)
    return {
        "score":  score,
        "passed": score >= 0.8
    }

evaluate({})                         # {"error": "empty response", "passed": False}
evaluate({"score": 0.91})            # {"score": 0.91, "passed": True}

# -------------------------------------------------------------- #
## Part 3 : try/except

# Simple single try/except block
def parse_score(raw: str) -> float:
    try:
        return float(raw)
    except ValueError:
        return 0.0

# Multiple except blocks
def load_eval_case(filepath: str) -> dict:
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"error": f"file not found: {filepath}"}
    except json.JSONDecodeError:
        return {"error": "invalid JSON in file"}
    except Exception as e: # Catch the error message with 'as e'
        return {"error": e}

# 'else' and 'finally'
try:
    result = json.loads(raw)
except json.JSONDecodeError:
    result = {}
else:
    print("parsing succeeded")      # runs only if NO exception occurred
finally:
    print("always runs")            # runs no matter what — success or failure


## Real AI Eval Pattern — Full Error Handling  
import json

def evaluate_case(case: dict) -> dict:

    # guard clause — catch empty input early
    if not case:
        return {"error": "empty case", "passed": False}

    try:
        # attempt to call LLM and parse response
        raw     = call_llm(case["input"])
        result  = json.loads(raw)
        score   = result.get("score", 0)
        return {
            "id":     case.get("id", "unknown"),
            "score":  score,
            "passed": score >= 0.8
        }

    except json.JSONDecodeError:
        return {"id": case.get("id"), "error": "invalid json", "passed": False}

    except KeyError as e:
        return {"id": case.get("id"), "error": f"missing key: {e}", "passed": False}

    except Exception as e:
        return {"id": case.get("id"), "error": f"unexpected: {e}", "passed": False}



True